# Práctica de temperatura en AeroStreamPeru
Ejecuta las celdas en orden. Arranca el puente y Wokwi antes de capturar el historial.
El identificador se configura con el script PowerShell de la práctica.

### 1. Configuración del flujo

In [ ]:
from pathlib import Path
import re
from urllib.request import Request, urlopen

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

CARPETA = Path("/opt/examen-temperatura")
CONFIGURACION = CARPETA / "identificador.local"
if not CONFIGURACION.is_file():
    raise FileNotFoundError("Configura la práctica con practica-temperatura.ps1 -Modo infra -Identificador ...")
ID = CONFIGURACION.read_text(encoding="ascii").strip()
if not re.fullmatch(r"[a-z]+", ID):
    raise ValueError("El identificador debe tener solo letras minúsculas.")
BROKER = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
TOPIC = f"{ID}-temperatura-eventos"
SENSOR = f"esp32-{ID}"
JOB = f"lambda26_{ID}_ml"
METRICA_REAL = "lambda26_ml_temperatura_real"
METRICA_SIGUIENTE = "lambda26_ml_temperatura_prediccion_siguiente"

IVY = Path("/opt/s08-streaming-estructurado/ivysettings.xml")
SALIDA = CARPETA / "artifacts" / ID
HISTORIAL = str(SALIDA / "historial")
MODELO = str(SALIDA / "modelo")
PREDICCIONES = str(SALIDA / "predicciones")
CHECKPOINT = str(SALIDA / "checkpoint")

spark = (
    SparkSession.builder.master("local[2]")
    .appName("PracticaTemperatura")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.jars.ivySettings", str(IVY))
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(TOPIC, SENSOR, JOB)

### 2. Historial en Parquet · S03

In [ ]:
ESQUEMA = """
tipoEvento STRING, sensorId STRING, temperatura DOUBLE,
origen STRING, timestamp LONG
"""

def leer_kafka(en_vivo=False):
    lector = spark.readStream if en_vivo else spark.read
    lector = (
        lector.format("kafka")
        .option("kafka.bootstrap.servers", BROKER)
        .option("subscribe", TOPIC)
        .option("startingOffsets", "latest" if en_vivo else "earliest")
    )
    if not en_vivo:
        lector = lector.option("endingOffsets", "latest")
    return (
        lector.load()
        .select("offset", F.from_json(F.col("value").cast("string"), ESQUEMA).alias("e"))
        .select("offset", "e.*")
        .filter(
            (F.col("tipoEvento") == "sensor.lectura")
            & (F.col("sensorId") == SENSOR)
            & (F.col("origen") == "wokwi")
            & F.col("temperatura").between(-40, 80)
            & (F.col("timestamp") > 0)
        )
    )

from time import time

fin_captura = int(time() * 1000)
inicio_captura = fin_captura - 10 * 60_000
ultimas = (
    leer_kafka()
    .filter(F.col("timestamp").between(inicio_captura, fin_captura))
    .orderBy(F.desc("timestamp"), F.desc("offset"))
    .limit(600)
)
ultimas.write.mode("overwrite").parquet(HISTORIAL)
historial = spark.read.parquet(HISTORIAL)
resumen = historial.agg(
    F.count("*").alias("lecturas"),
    F.min("timestamp").alias("primera_ms"),
    F.max("timestamp").alias("ultima_ms"),
    F.countDistinct("temperatura").alias("temperaturas_distintas"),
).first()
print(resumen.asDict())
assert resumen.lecturas >= 2, "Espera a que lleguen lecturas válidas de Wokwi."
minutos = (resumen.ultima_ms - resumen.primera_ms) / 60_000
print(f"Historial: {HISTORIAL} | lecturas: {resumen.lecturas} | {minutos:.2f} min")
print("Espera al menos 5 minutos de historial para el simulacro.")
assert minutos >= 4.98, "Deja que se acumule el mínimo y vuelve a ejecutar esta celda."
assert resumen.temperaturas_distintas > 1, "Mueve el control del DHT22."
historial.orderBy("timestamp", "offset").show(5, truncate=False)

### 3. Etiqueta, corte cronológico y modelo · S10

In [ ]:
orden = Window.orderBy("timestamp", "offset")
supervisado = (
    historial
    .withColumn("label", F.lead("temperatura", 1).over(orden))
    .filter(F.col("label").isNotNull())
    .withColumn("posicion", F.row_number().over(orden))
)
total = supervisado.count()
corte = int(total * 0.8)
assert corte >= 2 and total > corte, "Reúne más lecturas."
# El par en el límite usa como etiqueta la primera lectura de prueba.
train = supervisado.filter(F.col("posicion") < corte)
test = supervisado.filter(F.col("posicion") > corte)
print(f"Corte 80/20 en {corte}/{total}. Train: {train.count()}; test: {test.count()}.")
print("Se apartó el par que cruza el corte.")
supervisado.select("posicion", "timestamp", "offset", "temperatura", "label").show(8)
test.orderBy("posicion").select("posicion", "timestamp", "temperatura", "label").show(3)

evaluador = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
persistencia = test.withColumn("prediction", F.col("temperatura"))
print(f"RMSE persistencia: {evaluador.evaluate(persistencia):.4f} °C")

pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=["temperatura"], outputCol="features"),
    LinearRegression(featuresCol="features", labelCol="label", regParam=0.0),
])
modelo = pipeline.fit(train)
print(f"RMSE regresión: {evaluador.evaluate(modelo.transform(test)):.4f} °C")
modelo.write().overwrite().save(MODELO)
print(f"Modelo guardado: {MODELO}")

### 4. Inferencia y publicación · S10 y S11

In [ ]:
def publicar_metricas(temperatura, prediccion):
    cuerpo = (
        f"# TYPE {METRICA_REAL} gauge\n"
        f"{METRICA_REAL} {temperatura}\n"
        f"# TYPE {METRICA_SIGUIENTE} gauge\n"
        f"{METRICA_SIGUIENTE} {prediccion}\n"
    ).encode("utf-8")
    solicitud = Request(
        f"{PUSHGATEWAY}/metrics/job/{JOB}",
        data=cuerpo,
        method="PUT",
        headers={"Content-Type": "text/plain; version=0.0.4"},
    )
    with urlopen(solicitud, timeout=5):
        pass

def iniciar_inferencia(publicar=False):
    modelo_guardado = PipelineModel.load(MODELO)

    def por_lote(lote, numero):
        predicciones = modelo_guardado.transform(lote).select(
            "timestamp", "offset", "sensorId", "temperatura",
            F.col("prediction").alias("prediccion_siguiente"),
        ).persist()
        try:
            filas = predicciones.count()
            if filas == 0:
                return
            predicciones.write.mode("overwrite").parquet(f"{PREDICCIONES}/lote={numero}")
            print(f"Lote {numero}: {filas} predicciones")
            if publicar:
                ultima = predicciones.orderBy(F.desc("timestamp"), F.desc("offset")).first()
                try:
                    publicar_metricas(ultima.temperatura, ultima.prediccion_siguiente)
                except OSError as error:
                    print(f"No se publicó el lote {numero}: {error}")
        finally:
            predicciones.unpersist()

    return (
        leer_kafka(en_vivo=True).writeStream
        .foreachBatch(por_lote)
        .option("checkpointLocation", CHECKPOINT)
        .trigger(processingTime="5 seconds")
        .start()
    )

# Cambia a False para ensayar P3 por separado y a True para P4.
PUBLICAR_METRICAS = True
if "consulta" in globals() and consulta.isActive:
    consulta.stop()
consulta = iniciar_inferencia(publicar=PUBLICAR_METRICAS)
print(consulta.status)

### 5. Evidencia de P3 · Ejecuta tras esperar nuevos eventos

In [ ]:
print(consulta.status)
print(consulta.exception())
if Path(PREDICCIONES).exists():
    spark.read.parquet(PREDICCIONES).orderBy(
        F.desc("timestamp"), F.desc("offset")
    ).show(10, truncate=False)
else:
    print("Aún no hay lotes guardados. Deja Wokwi publicando y vuelve a ejecutar.")

### 6. Consultas para P4 y parada al terminar la demostración

In [ ]:
print(f'{METRICA_REAL}{{job="{JOB}"}}')
print(f'{METRICA_SIGUIENTE}{{job="{JOB}"}}')
# Cuando ya terminaste la demo:
# consulta.stop()